# SAR Soil Characterization — CDSE JupyterHub

Bu notebook CDSE JupyterHub üzerinde çalışmak için tasarlanmıştır.  
Hiçbir şey bilgisayara indirilmez — tüm işlem ve depolama CDSE sunucularında gerçekleşir.

**Başlamadan önce:**
1. `dataspace.copernicus.eu` → Analyse Data → JupyterHub
2. Terminal aç: `git clone https://github.com/sarpertaga/sar-soil-characterization`
3. Bu notebook'u aç ve hücreleri sırayla çalıştır

## 1 — Kurulum

In [ ]:
import subprocess, sys

# CDSE JupyterHub'da eksik olabilecek paketler
pkgs = [
    "whitebox>=2.3",
    "sentinelhub>=3.10",
    "python-dotenv>=1.0",
    "pyyaml>=6.0",
    "requests>=2.31",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# WhiteboxTools binary'sini indir (ilk çalıştırmada ~30 sn)
import whitebox
wbt = whitebox.WhiteboxTools()
print("WhiteboxTools:", wbt.version())

# Repoyu pip ile yükle (src/soilgeo import edilebilsin)
import os
REPO = os.path.expanduser("~/sar-soil-characterization")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO])
print("soilgeo paketi yüklendi")

## 2 — Credentials

CDSE Dashboard → User Settings → OAuth Clients → client_id ve client_secret'ı buraya gir.  
Bu değerler sadece oturum boyunca hafızada kalır, diske yazılmaz.

In [ ]:
import os
from getpass import getpass

os.environ["SH_CLIENT_ID"]     = getpass("SH_CLIENT_ID:     ")
os.environ["SH_CLIENT_SECRET"] = getpass("SH_CLIENT_SECRET: ")
print("Credentials set ✓")

## 3 — Config yükle

In [ ]:
from pathlib import Path
import sys

REPO = Path("~/sar-soil-characterization").expanduser()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from soilgeo.utils.config import load_aoi_config, load_config_dict

aoi = load_aoi_config(REPO / "config/aoi/konya.yml")
cfg_v1 = load_config_dict(REPO / "config/pipelines/v1.yml")
cfg_v2 = load_config_dict(REPO / "config/pipelines/v2.yml")

print(f"AOI: {aoi.name}  bbox: {aoi.bbox}")
print(f"CRS: {aoi.crs}  resolution: {aoi.resolution_m} m")

## 4 — V1 Pipeline

SAR + DEM → terrain → hydrology → NDDI → surface response classes

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, str(REPO / "pipelines/run_v1.py")],
    cwd=REPO, capture_output=False, text=True
)
print("V1 exit code:", result.returncode)

## 5 — V2 Pipeline

S2 bare-soil composite + SoilGrids + RF modelling → clay/sand/SOC haritaları

In [ ]:
# Aşamaları bağımsız çalıştır — her biri idempotent (tekrar çalıştırılabilir)
stages = ["fetch_s2", "s2_indices", "soilgrids", "features", "train", "predict", "catalog"]

for stage in stages:
    print(f"\n{'='*50}\nStage: {stage}\n{'='*50}")
    result = subprocess.run(
        [sys.executable, str(REPO / "pipelines/run_v2.py"), "--stages", stage],
        cwd=REPO, capture_output=False, text=True
    )
    if result.returncode != 0:
        print(f"HATA: {stage} başarısız")
        break
    print(f"{stage} ✓")

## 6 — Sonuçları görselleştir

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import rasterio

INTERIM   = REPO / "data/interim"
PROCESSED = REPO / "data/processed"
NODATA    = -9999.0

def load(path, band=1):
    with rasterio.open(path) as src:
        d = src.read(band).astype("float32")
        nd = src.nodata or NODATA
    return np.ma.masked_where(d == nd, d)

def ext(path):
    with rasterio.open(path) as src:
        b = src.bounds
    return [b.left, b.right, b.bottom, b.top]

def clip(a, lo=2, hi=98):
    p = np.nanpercentile(a.compressed(), [lo, hi])
    return np.clip(a, *p)

# ── V1 sonuçları ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor="#0d1117")
fig.suptitle("V1 — Feature Stack & Outputs", color="white", fontsize=13, fontweight="bold")

plots_v1 = [
    (INTERIM/"s1/s1_vv_konya_wet.tif",              "RdYlGn",   "VV Wet (dB)"),
    (INTERIM/"s1/s1_vv_konya_dry.tif",              "RdYlGn",   "VV Dry (dB)"),
    (INTERIM/"indices/nddi.tif",                    "RdBu",     "NDDI"),
    (INTERIM/"terrain/slope_resampled.tif",         "YlOrBr",   "Slope (°)"),
    (INTERIM/"hydrology/twi_resampled.tif",         "Blues",    "TWI"),
    (PROCESSED/"construction_risk_konya.tif",       "RdYlGn_r", "Construction Risk"),
]

for ax, (path, cmap, title) in zip(axes.flat, plots_v1):
    ax.set_facecolor("#161b22")
    if path.exists():
        im = ax.imshow(clip(load(path)), cmap=cmap, extent=ext(path), origin="upper")
        fig.colorbar(im, ax=ax, fraction=0.046).ax.tick_params(colors="gray", labelsize=7)
    ax.set_title(title, color="#e6edf3", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ── V2 sonuçları ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="#0d1117")
fig.suptitle("V2 — Soil Property Predictions (10 m)", color="white", fontsize=13, fontweight="bold")

plots_v2 = [
    (PROCESSED/"clay_10m_konya.tif", "YlOrBr", "Clay (%)",          0.1),
    (PROCESSED/"sand_10m_konya.tif", "OrRd",   "Sand (%)",          0.1),
    (PROCESSED/"soc_10m_konya.tif",  "YlGn",   "SOC (dg/kg)",       1.0),
]

for ax, (path, cmap, title, scale) in zip(axes, plots_v2):
    ax.set_facecolor("#161b22")
    if path.exists():
        data = load(path) * scale
        im = ax.imshow(clip(data), cmap=cmap, extent=ext(path), origin="upper")
        cb = fig.colorbar(im, ax=ax, fraction=0.046)
        cb.ax.tick_params(colors="gray", labelsize=7)
    else:
        ax.text(0.5, 0.5, "Pipeline henüz çalıştırılmadı",
                transform=ax.transAxes, ha="center", va="center", color="#58a6ff")
    ax.set_title(title, color="#e6edf3", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## 7 — Model metrikleri

In [ ]:
import json
metrics_path = PROCESSED / "v2_metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(f"{'Target':<8} {'RMSE':>10} {'R²':>8} {'MAE':>10}")
    print("-" * 40)
    for m in metrics:
        print(f"{m['target']:<8} {m['rmse_mean']:>10.3f} {m['r2_mean']:>8.3f} {m['mae_mean']:>10.3f}")
else:
    print("v2_metrics.json bulunamadı — önce train aşamasını çalıştır")